# Final report

**Gate:** Every number in the report traces to a JSON or CSV in reports/.

The narrative report for this brick is `reports/report.md`. The cells below load the saved artifacts; they do not retrain anything.

In [1]:
import json
from pathlib import Path
from IPython.display import Image, display, Markdown
import pandas as pd

ROOT = Path.cwd().parent
REPORTS = ROOT / "reports"
FIGURES = ROOT / "figures"

def md(name):
    p = REPORTS / name
    display(Markdown(p.read_text(encoding="utf-8"))) if p.exists() else print("missing", name)

def js(name, keys=None, indent=2):
    p = REPORTS / name
    if not p.exists():
        print("missing", name); return None
    d = json.loads(p.read_text(encoding="utf-8"))
    print(json.dumps({k: d[k] for k in keys} if keys else d, indent=indent)[:6000])
    return d


In [2]:
md('report.md')

# Drug-Food Interaction Prediction - Final Report

Three-class prediction (0 avoid, 1 beneficial/take with food, 2 non-significant) from a drug SMILES and a food-compound SMILES, following the paper's pipeline: MOE-type pair descriptors, four-step selection to 18 features, five baselines, tuned XGBoost, SHAP, held-out case studies, then two enhancement tracks. Model P (paper-like) applies a drug-level label to every compound; Mode S (specific) labels only compounds named by the interaction text. Mode S with Split G is the headline.

## 1. Data

- Raw tables: 1,516 drug rows, 23,950 interaction rows, 838,492 food rows; PubChem text found for 7 of 8 probe drugs.
- Drugs: 1,516 loaded, 1,437 parsed, 78 salt/parent duplicates removed, **1,359 kept** (1,350 with a DrugBank ID).
- Food compounds: 13,282 unique structures from 838,492 rows; 7,942 near-duplicates removed at Tanimoto 0.75; **5,022 kept** (4,877 organic, 145 inorganic).

## 2. Labels

- Mode P: **407,700 pairs**, classes {'0': 229500, '1': 65100, '2': 113100}.
- Mode S: **521,586 pairs**, classes {'0': 175024, '1': 176067, '2': 170495}.
- 7,045 interaction rows mapped to compounds in Mode S; the nine paper case pairs are held out of every training table (9 pairs).
- Sentence audit: 100 sampled rows, rule/annotation agreement on confirmed rows 0.60, rule coverage of 'Possible' rows 0.63. The structured FHDI labels disagree with literal text rules often enough that agreement is reported, not hidden.

## 3. Pair features

- 60 MOE-type descriptors per molecule over 1,359 drugs and 4,877 food compounds; pair vector = 3,600 products + 60 sums + 60 absolute differences + 60 ratios = **3,780 columns**.
- Verified against PyBioMed on 500 pairs: products max difference 5.00e-04, sums 1.14e-13; all values finite: True. The unit tests in tests/test_features.py pass (3 tests).

## 4. Splits

- Mode S: Split G drugs train/val/test = 225/167/55, pairs = 261,220/195,536/64,830; Split R pairs = 261,210/195,594/64,782. Split G drug overlaps = 0. Holdout pairs absent: True.
- Mode P: Split G drugs train/val/test = 680/510/169, pairs = 204,000/153,000/50,700; Split R pairs = 204,176/152,887/50,637. Split G drug overlaps = 0. Holdout pairs absent: True.
- External set: not obtained - Ryu et al. supplementary data was not downloaded in this build; the strict drug-grouped test partition plus the nine held-out case studies stand in, as allowed by the plan.

## 5. Feature selection

| step              |   Mode S |   Mode P |
|:------------------|---------:|---------:|
| start             |     3780 |     3780 |
| after_variance    |     3523 |     3524 |
| after_correlation |     1118 |     1065 |
| after_lasso       |      146 |      105 |
| after_rfecv       |      118 |       95 |
| rfe18             |       18 |       18 |

Stability (3 subsamples): Mode S 12/18 final features chosen in >=2 runs; Mode P 10/18.

## 6. Models

- Mode S: best CV model ExtraTrees (balanced=True) macro-F1 0.4919; tuned winner **XGBoost**, validation macro-F1 0.4799 after the full-training refit; XGBoost reload difference 0.0e+00.
- Mode P: best CV model XGBoost (balanced=True) macro-F1 0.3661; tuned winner **RandomForest**, validation macro-F1 0.3659 after the full-training refit; XGBoost reload difference 0.0e+00.

Baseline CV (Mode S, Split G grouped folds):

| model        | balanced   |   accuracy |   macro_f1 |   recall_0 |
|:-------------|:-----------|-----------:|-----------:|-----------:|
| XGBoost      | True       |    0.47162 |   0.470073 |   0.436003 |
| XGBoost      | False      |    0.47198 |   0.470495 |   0.474154 |
| LightGBM     | True       |    0.47082 |   0.468713 |   0.428241 |
| LightGBM     | False      |    0.47066 |   0.468398 |   0.4703   |
| RandomForest | True       |    0.47464 |   0.473644 |   0.462593 |
| RandomForest | False      |    0.47864 |   0.477355 |   0.47542  |
| ExtraTrees   | True       |    0.49244 |   0.491851 |   0.486595 |
| ExtraTrees   | False      |    0.49244 |   0.491726 |   0.489513 |
| MLP          | False      |    0.44522 |   0.44413  |   0.457363 |

## 7. Evaluation (frozen model, test partition)

| mode   | split   | model        |   test pairs |   test drugs |   accuracy |   macro-F1 |   recall_0 |   recall_1 |   recall_2 |
|:-------|:--------|:-------------|-------------:|-------------:|-----------:|-----------:|-----------:|-----------:|-----------:|
| S      | G       | XGBoost      |        61592 |           55 |     0.4477 |     0.4446 |     0.4119 |     0.3985 |     0.5352 |
| S      | R       | XGBoost      |        61488 |          447 |     0.7974 |     0.795  |     0.8537 |     0.8673 |     0.6716 |
| P      | G       | RandomForest |        50700 |          169 |     0.421  |     0.3794 |     0.5366 |     0.3648 |     0.267  |
| P      | R       | RandomForest |        50637 |         1359 |     0.6681 |     0.6427 |     0.6757 |     0.731  |     0.6164 |

Split R lets a drug appear in training and test, so it scores much higher. Split G is the number to quote.

## 8. SHAP

- Mode S (backend TreeExplainer): top features EstateVSA1*MTPSA (0.155), EstateVSA4*PEOEVSA8 (0.151), PEOEVSA8+PEOEVSA8 (0.139), PEOEVSA13+PEOEVSA13 (0.129), PEOEVSA7+PEOEVSA7 (0.125).
- Mode P (backend TreeExplainer): top features MRVSA5+MRVSA5 (0.072), PEOEVSA2+PEOEVSA2 (0.068), EstateVSA4+EstateVSA4 (0.054), EstateVSA5+EstateVSA5 (0.049), EstateVSA3*MRVSA5 (0.048).

## 9. Case studies (nine paper pairs held out, plus controls)

| pair                                       |   expected |   predicted | status        | verdict       |   confidence |   p_negative |   p_positive |   p_nonsignificant | sentence                                                                                                        | reason                                                                          | note                                         | disclaimer                                             |
|:-------------------------------------------|-----------:|------------:|:--------------|:--------------|-------------:|-------------:|-------------:|-------------------:|:----------------------------------------------------------------------------------------------------------------|:--------------------------------------------------------------------------------|:---------------------------------------------|:-------------------------------------------------------|
| tetracycline + calcium                     |          0 |           0 | rule          | pass          |   nan        |   nan        | nan          |         nan        | Tetracycline should not be taken with food containing Calcium.                                                  | Tetracyclines/quinolones bind calcium, iron and other cations; separate dosing. | paper case: chelation                        | Research tool; confirm with a physician or pharmacist. |
| methotrexate + ethanol                     |          0 |           0 | uncertain     | abstained     |     0.574361 |     0.574361 |   0.0809074  |           0.344732 | The model is not confident; ask a pharmacist before acting.                                                     | top probability 0.574 below 0.6                                                 | paper case: hepatotoxicity                   | Research tool; confirm with a physician or pharmacist. |
| diazepam + ethanol                         |          0 |           0 | uncertain     | abstained     |     0.447417 |     0.447417 |   0.134147   |           0.418436 | The model is not confident; ask a pharmacist before acting.                                                     | top probability 0.447 below 0.6                                                 | paper case: CNS depression                   | Research tool; confirm with a physician or pharmacist. |
| nitroglycerin + ethanol                    |          0 |           0 | uncertain     | abstained     |     0.442617 |     0.442617 |   0.14662    |           0.410763 | The model is not confident; ask a pharmacist before acting.                                                     | top probability 0.443 below 0.6                                                 | paper case: hypotension                      | Research tool; confirm with a physician or pharmacist. |
| digoxin + hyperforin                       |          0 |           0 | uncertain     | abstained     |     0.49417  |     0.49417  |   0.0753633  |           0.430466 | The model is not confident; ask a pharmacist before acting.                                                     | top probability 0.494 below 0.6                                                 | paper case: St John's Wort induction         | Research tool; confirm with a physician or pharmacist. |
| nisoldipine + bergamottin                  |          0 |           2 | uncertain     | abstained     |     0.465927 |     0.130788 |   0.403285   |           0.465927 | The model is not confident; ask a pharmacist before acting.                                                     | top probability 0.466 below 0.6                                                 | paper case: CYP3A4 inhibition                | Research tool; confirm with a physician or pharmacist. |
| midazolam + furanocoumarin                 |          0 |           2 | ok            | fail          |     0.618824 |     0.206599 |   0.174577   |           0.618824 | Midazolam may be taken with food containing Bergamottin.                                                        | nan                                                                             | paper case: furanocoumarin class             | Research tool; confirm with a physician or pharmacist. |
| warfarin + vitamin k1 2,3 epoxide          |          0 |           0 | ok            | pass          |     0.748348 |     0.748348 |   0.101176   |           0.150476 | Warfarin should not be taken with food containing CC(C)CCCC(C)CCCC(C)CCCC(=CCC12C(=O)C3=CC=CC=C3C(=O)C1(O2)C)C. | nan                                                                             | paper case: vitamin K antagonism             | Research tool; confirm with a physician or pharmacist. |
| warfarin + dimethyl disulfide              |          0 |           0 | ok            | pass          |     0.841933 |     0.841933 |   0.00871887 |           0.149349 | Warfarin should not be taken with food containing Dimethyl disulfide.                                           | nan                                                                             | paper case: garlic constituent               | Research tool; confirm with a physician or pharmacist. |
| warfarin + glucose                         |          2 |           0 | ok            | fail          |     0.855039 |     0.855039 |   0.0094709  |           0.13549  | Warfarin should not be taken with food containing C(C1C(C(C(C(O1)O)O)O)O)O.                                     | nan                                                                             | control: inert sugar                         | Research tool; confirm with a physician or pharmacist. |
| warfarin + unobtainium                     |        nan |         nan | not_supported | informational |   nan        |   nan        | nan          |         nan        | nan                                                                                                             | food compound not found or invalid SMILES                                       | control: compound absent from the food table | Research tool; confirm with a physician or pharmacist. |
| nitroglycerin + geranyl rhamnosylglucoside |        nan |         nan | not_supported | informational |   nan        |   nan        | nan          |         nan        | nan                                                                                                             | food compound not found or invalid SMILES                                       | paper unseen compound                        | Research tool; confirm with a physician or pharmacist. |
| warfarin + dicoumarol                      |          0 |           0 | ok            | pass          |     0.605801 |     0.605801 |   0.0775405  |           0.316658 | Warfarin should not be taken with food containing C1=CC=C2C(=C1)C(=C(C(=O)O2)CC3=C(C4=CC=CC=C4OC3=O)O)O.        | nan                                                                             | paper unseen compound                        | Research tool; confirm with a physician or pharmacist. |

## 10. Ablations and enhancements (Mode S, Split G)

| drug_only                                                                                                                                                                                                                                                                                                                                                 | food_only                                                                                                                                                                                                                                                                                                                                               | shuffled_food                                                                                                                                                                                                                                                                                                                                                | all_3780_features                                                                                                                                                                                                                                                                                                                                   | 18+E1a_drug_metadata                                                                                                                                                                                                                                                                                                                                     | 18+E1b_food_class                                                                                                                                                                                                                                                                                                                                      | 18+E1c_fingerprints                                                                                                                                                                                                                                                                                                                                      | 18+E1abc_all                                                                                                                                                                                                                                                                                                                                           | 18_moe                                                                                                                                                                                                                                                                                                                                                   |
|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| 0.3461650863748539                                                                                                                                                                                                                                                                                                                                        | 0.5443401740485777                                                                                                                                                                                                                                                                                                                                      | 0.376461228731004                                                                                                                                                                                                                                                                                                                                            | 0.5048666666666667                                                                                                                                                                                                                                                                                                                                  | 0.4467138589427198                                                                                                                                                                                                                                                                                                                                       | 0.4440998831017015                                                                                                                                                                                                                                                                                                                                     | 0.4756624236913885                                                                                                                                                                                                                                                                                                                                       | 0.47351928821924927                                                                                                                                                                                                                                                                                                                                    | 0.4522178204961683                                                                                                                                                                                                                                                                                                                                       |
| 0.2454404647536165                                                                                                                                                                                                                                                                                                                                        | 0.5296357270087099                                                                                                                                                                                                                                                                                                                                      | 0.376106749073442                                                                                                                                                                                                                                                                                                                                            | 0.4946006227620976                                                                                                                                                                                                                                                                                                                                  | 0.4448661292353524                                                                                                                                                                                                                                                                                                                                       | 0.4420841652201412                                                                                                                                                                                                                                                                                                                                     | 0.44822258727787195                                                                                                                                                                                                                                                                                                                                      | 0.4485509302452495                                                                                                                                                                                                                                                                                                                                     | 0.45014237972466775                                                                                                                                                                                                                                                                                                                                      |
| {'0': {'precision': 0.3895790200138026, 'recall': 0.06512459621596678, 'f1': 0.11159434615004447, 'support': 17336}, '1': {'precision': 0.80758137618459, 'recall': 0.08239102106015385, 'f1': 0.14952700640830027, 'support': 23789}, '2': {'precision': 0.3240265164305899, 'recall': 0.8907998241070992, 'f1': 0.47520004170250474, 'support': 20467}} | {'0': {'precision': 0.33507272845867847, 'recall': 0.2963197969543147, 'f1': 0.3145069948265834, 'support': 17336}, '1': {'precision': 0.5246668461434917, 'recall': 0.6553869435453361, 'f1': 0.5827866554527614, 'support': 23789}, '2': {'precision': 0.7735871864611665, 'recall': 0.6253481213661015, 'f1': 0.6916135307467848, 'support': 20467}} | {'0': {'precision': 0.3374339109040465, 'recall': 0.37182741116751267, 'f1': 0.3537967562227284, 'support': 17336}, '1': {'precision': 0.48959901845965087, 'recall': 0.36903610912606666, 'f1': 0.42085330776605945, 'support': 23789}, '2': {'precision': 0.3242120693867579, 'recall': 0.38901646552987734, 'f1': 0.35367018323153804, 'support': 20467}} | {'0': {'precision': 0.44986449864498645, 'recall': 0.38757879990660754, 'f1': 0.4164053681173962, 'support': 4283}, '1': {'precision': 0.6304460580912863, 'recall': 0.4281437125748503, 'f1': 0.5099643381581708, 'support': 5678}, '2': {'precision': 0.4671317413469278, 'recall': 0.691010121055765, 'f1': 0.557432162010726, 'support': 5039}} | {'0': {'precision': 0.36919075428178827, 'recall': 0.433952468850946, 'f1': 0.39896057062551377, 'support': 17336}, '1': {'precision': 0.5646863512391794, 'recall': 0.4003531043759721, 'f1': 0.4685278563522322, 'support': 23789}, '2': {'precision': 0.42987391679329745, 'recall': 0.5114086089803098, 'f1': 0.4671099607283113, 'support': 20467}} | {'0': {'precision': 0.36812078886539124, 'recall': 0.43176049838486386, 'f1': 0.3974090100613236, 'support': 17336}, '1': {'precision': 0.5528868906767629, 'recall': 0.3928706545041826, 'f1': 0.4593419015555501, 'support': 23789}, '2': {'precision': 0.43202627797166904, 'recall': 0.514095861630918, 'f1': 0.46950158404355, 'support': 20467}} | {'0': {'precision': 0.5077567336061674, 'recall': 0.30774111675126903, 'f1': 0.38322019897281184, 'support': 17336}, '1': {'precision': 0.5721942504466461, 'recall': 0.2961873134642061, 'f1': 0.39032767360053183, 'support': 23789}, '2': {'precision': 0.4363054860591679, 'recall': 0.826501197048908, 'f1': 0.5711198892602721, 'support': 20467}} | {'0': {'precision': 0.5443668993020937, 'recall': 0.3464467005076142, 'f1': 0.423419930205506, 'support': 17336}, '1': {'precision': 0.5514225128114508, 'recall': 0.2623481441002144, 'f1': 0.35554162987438404, 'support': 23789}, '2': {'precision': 0.4311307051298387, 'recall': 0.8265989153271119, 'f1': 0.5666912306558585, 'support': 20467}} | {'0': {'precision': 0.3740492393915132, 'recall': 0.43118366405168435, 'f1': 0.40058949624866025, 'support': 17336}, '1': {'precision': 0.5710159995379195, 'recall': 0.4155702215309597, 'f1': 0.48104715099021944, 'support': 23789}, '2': {'precision': 0.431858407079646, 'recall': 0.512630087457859, 'f1': 0.46879049193512357, 'support': 20467}} |
| [[1129, 0, 16207], [1, 1960, 21828], [1768, 467, 18232]]                                                                                                                                                                                                                                                                                                  | [[5137, 9549, 2650], [7102, 15591, 1096], [3092, 4576, 12799]]                                                                                                                                                                                                                                                                                          | [[6446, 4177, 6713], [5127, 8779, 9883], [7530, 4975, 7962]]                                                                                                                                                                                                                                                                                                 | [[1660, 761, 1862], [1137, 2431, 2110], [893, 664, 3482]]                                                                                                                                                                                                                                                                                           | [[7523, 4122, 5691], [6074, 9524, 8191], [6780, 3220, 10467]]                                                                                                                                                                                                                                                                                            | [[7485, 4209, 5642], [6252, 9346, 8191], [6596, 3349, 10522]]                                                                                                                                                                                                                                                                                          | [[5335, 3469, 8532], [3420, 7046, 13323], [1752, 1799, 16916]]                                                                                                                                                                                                                                                                                           | [[6006, 3457, 7873], [3098, 6241, 14450], [1929, 1620, 16918]]                                                                                                                                                                                                                                                                                         | [[7475, 4051, 5810], [5910, 9886, 7993], [6599, 3376, 10492]]                                                                                                                                                                                                                                                                                            |
| 0.5                                                                                                                                                                                                                                                                                                                                                       | 0.9                                                                                                                                                                                                                                                                                                                                                     | 1.4                                                                                                                                                                                                                                                                                                                                                          | 281.8                                                                                                                                                                                                                                                                                                                                               | 1.4                                                                                                                                                                                                                                                                                                                                                      | 1.4                                                                                                                                                                                                                                                                                                                                                    | 5.6                                                                                                                                                                                                                                                                                                                                                      | 5.7                                                                                                                                                                                                                                                                                                                                                    | 2.7                                                                                                                                                                                                                                                                                                                                                      |
| nan                                                                                                                                                                                                                                                                                                                                                       | nan                                                                                                                                                                                                                                                                                                                                                     | nan                                                                                                                                                                                                                                                                                                                                                          | 15000                                                                                                                                                                                                                                                                                                                                               | nan                                                                                                                                                                                                                                                                                                                                                      | nan                                                                                                                                                                                                                                                                                                                                                    | nan                                                                                                                                                                                                                                                                                                                                                      | nan                                                                                                                                                                                                                                                                                                                                                    | nan                                                                                                                                                                                                                                                                                                                                                      |
| nan                                                                                                                                                                                                                                                                                                                                                       | nan                                                                                                                                                                                                                                                                                                                                                     | nan                                                                                                                                                                                                                                                                                                                                                          | nan                                                                                                                                                                                                                                                                                                                                                 | 19                                                                                                                                                                                                                                                                                                                                                       | 28                                                                                                                                                                                                                                                                                                                                                     | 530                                                                                                                                                                                                                                                                                                                                                      | 541                                                                                                                                                                                                                                                                                                                                                    | nan                                                                                                                                                                                                                                                                                                                                                      |

E2 conformal at a 90% target: test coverage 0.887, abstention rate 0.895, selective accuracy 0.585 on 6,461 pairs. Drug-level bootstrap 95% CI for macro-F1: [0.355, 0.465].

Food-side descriptors alone beat the 18 pair features here; shuffled pairing costs about 0.07 macro-F1. Both say the signal is dominated by compound identity, which is exactly what Mode S's weak class-2 pairs predict.

## 11. Decisions where the paper was ambiguous

- Correlated-pair tie-break: keep the feature with the higher ANOVA F-score (Mode S LASSO kept 146 features; L1 multinomial logistic kept 86, overlap 86).
- Three-class LASSO: regression on codes 0/1/2 (paper-like), cross-checked with L1 multinomial logistic.
- RFECV chose 118 features in Mode S and 95 in Mode P, so the final set uses RFE fixed at 18; both scores are reported.
- Label rules are applied only to the 'Possible' interaction rows; confirmed structured labels are kept as-is.
- Model selection uses validation macro-F1, never test, unlike the paper's reported procedure.

Why these numbers differ from the paper:

- Labels come from FHDI structured annotations, not DrugBank sentences; the paper's 97% accuracy is not reproducible on this label source.
- Mode S removes the shortcut of labelling a whole drug row uniformly, so scores drop sharply.
- Split G holds out entire drugs, so the model cannot memorise drug identity.
- Training used a 50,000-pair cap for CV/tuning and a 40,000-pair cap for the ablation stage because of the 3.7 GB machine, then refit on the full training partition.

## 12. Limitations and safety

- This is decision support research, not medical advice; every recommendation carries the disclaimer.
- Inorganic minerals bypass the model through a name-based rule layer; out-of-domain molecules are refused.
- The external Ryu et al. benchmark was not obtained; Split G test plus held-out case studies stand in for it.

## 13. Reproduce

```bash
bash scripts/run_all.sh        # bricks 1-12, both modes, then notebooks
python scripts/make_report.py  # rebuild this report from reports/*
```


In [3]:
# Reproduce this brick from scratch (long running):
# !cd .. && PYTHONPATH=src python src/brick13_*.py